In [1]:
from nnsight import LanguageModel
from nnsight.models.UnifiedTransformer import UnifiedTransformer

import sys

sys.path.append("../../../")

from nngine import alter

import torch 
import einops

model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)
# alter(model)

tokenizer = model.tokenizer

tl_model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device="cuda",
)

tl_model.set_use_hook_mlp_in(True)
tl_model.set_use_split_qkv_input(True)
tl_model.set_use_attn_result(True)

def test_resolution(ground_truth, sample):
    for resolution in [1e-12, 1e-10, 1e-6, 1e-4, 1e-3, 1e-2, 1.0]:
        pct = (sample - ground_truth > resolution).float().mean().item()
        print(f'pct out of range {resolution:.1e} = {pct:.2%}')

Loaded pretrained model gpt2-small into HookedTransformer


# TESTING ATTENTION RESULT (GOOD)

In [2]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

with model.trace(sample):
    c_proj = model.transformer.h[8].attn.c_proj

    heads = einops.rearrange(
        c_proj.input[0][0], 
        "batch seq (head_idx head_dim) -> batch seq head_idx head_dim", 
        head_idx=12, 
        head_dim=64
    )

    w_o_split = einops.rearrange(
        c_proj.weight,
        "(head_idx head_dim) d_model \
            -> head_idx head_dim d_model",
        head_idx=12,
        head_dim=64
    )

    attn_out = einops.einsum(
        heads, w_o_split,
        "batch pos head_idx head_dim, head_idx head_dim d_model -> batch pos head_idx d_model",
    )

    attn_out.save()

    logits = model.output.logits[:,-1,:]

    logits.save()
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[8].attn.hook_result.output.save()

    tl_logits = tl_model.unembed.output[:,-1,:]

    tl_logits.save()

    value = tl_logits.sum()
    value.backward()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [3]:
test_resolution(tl_test, attn_out)

pct out of range 1.0e-12 = 48.31%
pct out of range 1.0e-10 = 48.31%
pct out of range 1.0e-06 = 0.19%
pct out of range 1.0e-04 = 0.00%
pct out of range 1.0e-03 = 0.00%
pct out of range 1.0e-02 = 0.00%
pct out of range 1.0e+00 = 0.00%


# TESTING MLP GRAD
- ok after visual inspection 

In [4]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

with model.trace(sample):
    test = model.transformer.h[0].mlp.input[0][0].grad.save()

    logits = model.output.logits[:,-1,:]

    logits.save()
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].mlp.input[0][0].grad.save()

    tl_logits = tl_model.unembed.output[:,-1,:]

    tl_logits.save()

    value = tl_logits.sum()
    value.backward()

In [5]:
test

tensor([[[ 19692.0625,  12324.0420,  21004.7812,  ...,  17840.5352,
           14971.1729, -18485.4844],
         [-10719.5195,   3828.1494,  15632.2051,  ...,   7288.0049,
            9257.4961,  25204.1270],
         [  9393.4883, -14390.8691,  -1926.5837,  ...,   5023.5786,
           10999.0400,  19980.4648],
         ...,
         [ 25940.8184,  -8910.6191,  30141.5156,  ..., -18615.6797,
          -13897.0342,  25587.3086],
         [-33996.0703, -11709.2988,  -1442.7485,  ...,   7854.1221,
            1280.2451, -19651.1504],
         [-60986.6875,  11353.3613,  -3414.4941,  ...,  33537.3828,
           14872.0000,   -883.0283]]], device='cuda:0')

In [6]:
tl_test

tensor([[[ 19692.0625,  12324.0107,  21004.7773,  ...,  17840.5195,
           14971.1299, -18485.4531],
         [-10719.5215,   3828.1543,  15632.1396,  ...,   7288.0454,
            9257.4688,  25204.1250],
         [  9393.4746, -14390.8477,  -1926.5769,  ...,   5023.5532,
           10998.9766,  19980.4082],
         ...,
         [ 25940.7500,  -8910.5801,  30141.4453,  ..., -18615.6035,
          -13897.0703,  25587.2773],
         [-33996.0273, -11709.2969,  -1442.7305,  ...,   7854.1260,
            1280.2354, -19651.0684],
         [-60986.6055,  11353.3477,  -3414.5400,  ...,  33537.4062,
           14872.0000,   -882.9482]]], device='cuda:0')

In [7]:
test_resolution(tl_test, test)

pct out of range 1.0e-12 = 51.24%
pct out of range 1.0e-10 = 51.24%
pct out of range 1.0e-06 = 51.24%
pct out of range 1.0e-04 = 51.24%
pct out of range 1.0e-03 = 49.72%
pct out of range 1.0e-02 = 35.64%
pct out of range 1.0e+00 = 0.00%


# SPLIT Q INPUT

In [8]:
sample = tokenizer.encode("Susan and Mary went to the store, Susan went in and")

slice_index = 0

with model.trace(sample):

    attention = model.transformer.h[0].attn
    attn_input = attention.input[0][0]


    repeated_attn = einops.repeat(
        attn_input,
        "batch pos d_model -> batch pos head_idx d_model",
        head_idx=12,
    ).clone()

    repeated_attn.grad.save()
    
    weight = torch.tensor_split(attention.c_attn.weight, 3, dim=1)[slice_index]
    split_weight = einops.rearrange(
        weight, 
        "d_model (head_index d_head) -> head_index d_model d_head",
        head_index=12
    )

    split_bias = einops.rearrange(
        attention.c_attn.bias,
        "(qkv head_idx d_head) -> qkv head_idx d_head",
        qkv=3,
        head_idx=12,
        d_head=64,
    )[slice_index]
    
    split_out = einops.einsum(
        repeated_attn, split_weight,
        "batch pos head_idx d_model, head_idx d_model d_head -> batch pos head_idx d_head",
    ) + split_bias
    
    split_attn = list(attention.c_attn.output.split(768, dim=2))

    split_attn[slice_index] = einops.rearrange(
       split_out, 
        "batch pos head_idx d_head -> batch pos (head_idx d_head)",
    )

    attention.c_attn.output = torch.cat(split_attn, dim=2)
    
    logits = model.output.logits[:,-1,:]

    value = logits.sum()
    value.backward()

# with tl_model.trace(sample):
#     tl_test = tl_model.blocks[0].mlp.input[0][0].grad.save()

#     tl_logits = tl_model.unembed.output[:,-1,:]

#     tl_logits.save()

#     value = tl_logits.sum()
#     value.backward()

In [9]:
repeated_attn

ValueError: Accessing Proxy value before it's been set.